# Running ComfyUI with Cloudflare Tunnel

This notebook is a smoother Cloudflare-based version of the original ComfyUI setup. It installs ComfyUI, starts it locally, and exposes it through a Cloudflare tunnel instead of Pinggy.

Steps:
1. Run the cells below in order.
2. If Cloudflare prompts for login, complete the browser flow once.
3. Copy the generated public URL from the tunnel log.


In [ ]:
# Install required system packages
!apt-get update -qq
!apt-get install -y -qq wget curl libgl1


In [ ]:
# Clone ComfyUI into the workspace
%cd /content
!rm -rf ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI


In [ ]:
# Install Python dependencies for ComfyUI
!pip install -r requirements.txt

In [ ]:
# Install the Cloudflare tunnel client
!curl -L --fail https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Start ComfyUI in the background
import os, subprocess, time
os.chdir('/content/ComfyUI')
log_path = '/content/comfyui.log'
with open(log_path, 'wb') as out:
    subprocess.Popen(['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'], stdout=out, stderr=out, start_new_session=True)
print('ComfyUI start command launched. Waiting 20 seconds...')
time.sleep(20)
print('Latest ComfyUI log output:')
!tail -n 30 /content/comfyui.log

In [ ]:
# Start a Cloudflare tunnel pointing to the local ComfyUI web server
!nohup cloudflared tunnel --url http://127.0.0.1:8188 --no-autoupdate >/content/cloudflared.log 2>&1 &
print('Tunnel command launched. Waiting 10 seconds for URL creation...')
import time
time.sleep(10)
!tail -n 80 /content/cloudflared.log

In [ ]:
# Try to detect the generated public URL from the tunnel log
import os, re, time
log_file = '/content/cloudflared.log'
for _ in range(30):
    if os.path.exists(log_file):
        with open(log_file, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        match = re.search(r'https?://[^\s"]+', text)
        if match:
            print('Public URL detected:')
            print(match.group(0))
            break
        if 'Your Quick Tunnel has been created' in text or 'trycloudflare.com' in text:
            print(text)
            break
    time.sleep(3)
else:
    print('No public URL detected yet. Check /content/cloudflared.log manually.')


## Notes

- Cloudflare can create a temporary public URL automatically via a browser login flow.
- If you want to use a named tunnel instead, run a setup like this in a separate terminal: 

```bash
cloudflared tunnel login
cloudflared tunnel create my-tunnel
cloudflared tunnel run my-tunnel --url http://127.0.0.1:8188
```

Then open the URL produced by your tunnel.
